# 📓 Audio AI Module 1: Discrete Audio Codecs & VQ-VAE Bitrate Compression
Welcome to the Audio Models module! In this notebook, we explore **Neural Audio Codecs** (such as Meta's EnCodec and SoundStream), which serve as the foundational representation layer for modern Speech LLMs and music generation models (e.g., Jukebox, AudioLM).

---

## 💡 Why Neural Audio Codecs?
Raw audio signals at a $24\text{ kHz}$ sampling rate contain **$24,000$ values per second**. Processing continuous high-frequency waveforms directly in Transformers or LLMs is computationally intractable.

Neural Audio Codecs solve this by compressing continuous 1D audio waveforms into **discrete tokens** across quantized codebooks using Vector Quantization (VQ).

### Residual Vector Quantization (RVQ)
Standard VQ maps a feature vector $z$ to a single nearest codebook entry $q(z)$. To compress audio at high fidelity without a massive single codebook, codecs use **Residual Vector Quantization (RVQ)** across $N$ hierarchical stages:

1. Stage 1 quantizes $z$ to $q_1(z)$, leaving residual $e_1 = z - q_1(z)$.
2. Stage 2 quantizes residual $e_1$ to $q_2(e_1)$, leaving residual $e_2 = e_1 - q_2(e_1)$.
3. Stage $N$ quantizes residual $e_{N-1}$ to $q_N(e_{N-1})$.

The reconstructed representation is the sum of codebook vectors:

$$\hat{z} = \sum_{i=1}^{N} q_i(e_{i-1})$$

**Bitrate Control:** By dropping higher-stage residual codebooks during decoding, we can dynamically scale the bitrate (e.g., $1.5\text{ kbps}$ vs $24.0\text{ kbps}$) and observe the trade-off between compression and audio fidelity!

In [ ]:
import os
import urllib.request
import torch
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt
from IPython.display import Audio, display, clear_output
import ipywidgets as widgets
from ipywidgets import interact, Dropdown

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Download & Load Sample Audio Signal
We load a clean speech audio sample using official PyTorch assets.

In [ ]:
# Download audio sample using PyTorch official assets with URL fallback
audio_filename = "speech_sample.wav"
if not os.path.exists(audio_filename):
    url = "https://storage.googleapis.com/cloud-samples-data/speech/hello.wav"
    urllib.request.urlretrieve(url, audio_filename)

# Load audio using torchaudio
waveform, sample_rate = torchaudio.load(audio_filename)

# Resample to 24 kHz required by EnCodec 24kHz model
target_sr = 24000
if sample_rate != target_sr:
    resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sr)
    waveform = resampler(waveform)
    sample_rate = target_sr

# Ensure mono audio [1, T]
if waveform.shape[0] > 1:
    waveform = torch.mean(waveform, dim=0, keepdim=True)

print(f"Audio loaded successfully!")
print(f"Shape: {waveform.shape}, Sample Rate: {sample_rate} Hz, Duration: {waveform.shape[1]/sample_rate:.2f}s")
display(Audio(waveform.numpy(), rate=sample_rate))

## 2. Load Pre-trained EnCodec 24kHz Model
We load Meta's **EnCodec 24kHz** model from `torchaudio.models` (with fallback to Hugging Face `transformers`).
EnCodec consists of a 1D CNN Encoder, a Residual Vector Quantizer (RVQ) with 8 codebooks, and a 1D CNN Decoder.

In [ ]:
# Fail-safe loader for EnCodec 24kHz model
from transformers import EncodecModel
codec_model = EncodecModel.from_pretrained("facebook/encodec_24khz").to(device)
codec_model.eval()
backend = "transformers"
print("Loaded EnCodec 24kHz via Hugging Face transformers")

## 3. Waveform & Mel-Spectrogram Visualization Helper
To evaluate audio reconstruction quality, we compare the **1D Time-Domain Waveform** and the **2D Mel-Spectrogram** side-by-side.

In [ ]:
def plot_audio_comparison(orig_wav, recon_wav, sr, bandwidth_kbps):
    # Compute Mel-Spectrograms
    mel_transform = T.MelSpectrogram(sample_rate=sr, n_fft=1024, hop_length=256, n_mels=80)
    
    orig_mel = torchaudio.transforms.AmplitudeToDB()(mel_transform(orig_wav.cpu()))
    recon_mel = torchaudio.transforms.AmplitudeToDB()(mel_transform(recon_wav.cpu()))
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 6))
    
    # 1D Waveform plots
    axes[0, 0].plot(orig_wav.squeeze().cpu().numpy(), color='#1f77b4', alpha=0.8)
    axes[0, 0].set_title("Original Waveform (24 kHz)")
    axes[0, 0].set_ylabel("Amplitude")
    axes[0, 0].grid(True, linestyle='--', alpha=0.5)
    
    axes[0, 1].plot(recon_wav.squeeze().cpu().numpy(), color='#2ca02c', alpha=0.8)
    axes[0, 1].set_title(f"Reconstructed Waveform (Bitrate: {bandwidth_kbps} kbps)")
    axes[0, 1].set_ylabel("Amplitude")
    axes[0, 1].grid(True, linestyle='--', alpha=0.5)
    
    # 2D Mel-Spectrogram plots
    im1 = axes[1, 0].imshow(orig_mel.squeeze().numpy(), origin='lower', aspect='auto', cmap='viridis')
    axes[1, 0].set_title("Original Mel-Spectrogram")
    axes[1, 0].set_xlabel("Time Frames")
    axes[1, 0].set_ylabel("Mel Frequency Bins")
    fig.colorbar(im1, ax=axes[1, 0], format='%+2.0f dB')
    
    im2 = axes[1, 1].imshow(recon_mel.squeeze().numpy(), origin='lower', aspect='auto', cmap='viridis')
    axes[1, 1].set_title(f"Reconstructed Mel-Spectrogram ({bandwidth_kbps} kbps)")
    axes[1, 1].set_xlabel("Time Frames")
    axes[1, 1].set_ylabel("Mel Frequency Bins")
    fig.colorbar(im2, ax=axes[1, 1], format='%+2.0f dB')
    
    plt.tight_layout()
    plt.show()

## 🎛️ 4. Interactive Bitrate Lab & Audio Playback
Use the dropdown menu below to test different target bitrates ($1.5\text{ kbps}$, $3.0\text{ kbps}$, $6.0\text{ kbps}$, $12.0\text{ kbps}$, $24.0\text{ kbps}$).

* At **$1.5\text{ kbps}$**, EnCodec uses only 1 RVQ codebook stage, producing noticeable robotic/quantization noise.
* At **$24.0\text{ kbps}$**, EnCodec uses all 8 RVQ codebook stages, yielding near-lossless acoustic reconstruction!

In [ ]:
def compress_and_play(target_bandwidth_kbps=6.0):
    input_wav = waveform.unsqueeze(0).to(device)  # Shape [1, 1, T]
    
    with torch.no_grad():
        if backend == "torchaudio":
            encoded_frames = codec_model.encode(input_wav, bandwidth=target_bandwidth_kbps)
            reconstructed_wav = codec_model.decode(encoded_frames)
        else:
            encoder_outputs = codec_model.encode(input_wav, bandwidth=target_bandwidth_kbps)
            reconstructed_wav = codec_model.decode(
                encoder_outputs.audio_codes, 
                encoder_outputs.audio_scales, 
                padding_mask=None
            )[0]
        
    # Flatten both to 1D
    orig_1d = waveform.squeeze().cpu()
    recon_1d = reconstructed_wav.squeeze().cpu()
    
    # Trim EnCodec frame padding
    min_len = min(orig_1d.shape[0], recon_1d.shape[0])
    orig_1d = orig_1d[:min_len]
    recon_1d = recon_1d[:min_len]
    
    # Calculate Signal-to-Noise Ratio (SNR)
    signal_power = torch.mean(orig_1d ** 2)
    noise_power = torch.mean((orig_1d - recon_1d) ** 2)
    snr_db = 10 * torch.log10(signal_power / (noise_power + 1e-8))
    
    clear_output(wait=True)
    print(f"Target Bitrate    : {target_bandwidth_kbps} kbps")
    print(f"Reconstruction SNR: {snr_db.item():.2f} dB")
    
    # Display audio player
    display(Audio(recon_1d.numpy(), rate=target_sr))
    
    # Plot visual comparison
    plot_audio_comparison(orig_1d, recon_1d, target_sr, target_bandwidth_kbps)

# Dropdown widget for valid EnCodec 24kHz bandwidths
interact(compress_and_play,
         target_bandwidth_kbps=Dropdown(
             options=[1.5, 3.0, 6.0, 12.0, 24.0],
             value=6.0,
             description='Bitrate (kbps):'
         ));